# Teammate 2 — AR25 game exploration

**Goal:** understand how ARC-AGI-3 game `ar25-0c556536` reacts to each action, using controlled experiments only
(no hidden source code, no walkthroughs).

**How this notebook is organised**

| Section | What it does |
|---|---|
| 0. Setup | Imports, paths, helpers. Every run writes to a **new** folder `data/raw/exploration/<RUN_ID>/` so raw data is never overwritten. |
| 1. Initial screen | Loads AR25, displays the first frame, lists the objects found on screen. |
| 2. Experiment runner | Every experiment starts from `env.reset()`, then applies a fixed action sequence and saves before/after frames for **every** step. |
| 3. Controlled experiments | E01–E07, each answering one question. |
| 4. Observations vs hypotheses | Facts measured automatically are kept separate from hypotheses, which have an explicit status. |
| 5. Unsuccessful experiments | Experiments with no visible effect, an error or a game over are logged separately. |
| 6. Summary & report | Writes `reports/teammate_2_exploration_report.md`. |

**Run it:** from the repository root, `jupyter notebook notebooks/teammate_2_exploration.ipynb`, then *Run All*.
Requires the same environment as `scripts/13_collect_sequence.py` (`arc_agi`, `arcengine`, `numpy`, `matplotlib`).

**Conventions:** frames are 2-D integer grids (colour index per cell). For `ACTION6`, `x` = column, `y` = row
(same as `scripts/07_collect_transition.py`). When a step returns several frames (animation), the **last** frame is
used as the resulting state; all frames are still saved.

## 0. Setup

In [ ]:
import json
import platform
from collections import Counter, deque
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from arc_agi import Arcade
from arcengine import GameAction

GAME_ID = "ar25-0c556536"
SEED = 0
RANDOM_SEED = 42          # used only by experiment E07
TERMINAL_STATES = {"WIN", "GAME_OVER"}

# Locate repository root whether the notebook is started from the root or from notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "scripts").exists():
    raise RuntimeError(f"Start Jupyter from the repository root (current: {ROOT})")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = ROOT / "data" / "raw" / "exploration" / RUN_ID
FRAMES_DIR = OUT / "frames"
IMAGES_DIR = OUT / "images"
REPORTS_DIR = ROOT / "reports"
for d in (FRAMES_DIR, IMAGES_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXPERIMENTS_LOG = OUT / "experiments.jsonl"
FAILED_LOG = OUT / "unsuccessful_experiments.jsonl"

print("Run ID     :", RUN_ID)
print("Output dir :", OUT.relative_to(ROOT))
print("Python     :", platform.python_version(), "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)

In [ ]:
# ---------- display helpers ----------
# 16-colour palette approximating the ARC-AGI-3 renderer (display only; analysis uses raw integers).
PALETTE = ["#FFFFFF", "#CCCCCC", "#999999", "#666666", "#333333", "#000000", "#E53AA3", "#FF7BCC",
           "#F93C31", "#1E93FF", "#88D8F1", "#FFDC00", "#FF851B", "#921231", "#4FCC30", "#A356D6"]
CMAP = ListedColormap(PALETTE)


def last_frame(obs):
    return np.array(obs.frame[-1])


def state_name(obs):
    s = obs.state
    return getattr(s, "value", None) or getattr(s, "name", None) or str(s)


def show_frame(frame, ax, title=""):
    ax.imshow(frame, cmap=CMAP, vmin=0, vmax=15, interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.set_xticks(range(0, frame.shape[1], 8))
    ax.set_yticks(range(0, frame.shape[0], 8))
    ax.tick_params(labelsize=6)


def show_diff(before, after, ax, title="changed cells"):
    ax.imshow(before, cmap=CMAP, vmin=0, vmax=15, interpolation="nearest", alpha=0.35)
    mask = np.ma.masked_where(before == after, np.ones_like(before))
    ax.imshow(mask, cmap=ListedColormap(["#FF00FF"]), interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.tick_params(labelsize=6)


# ---------- analysis helpers ----------
# Observed in run 20261120T012831Z: column x=63 is a status bar whose cells turn from colour 11 to 5,
# one per action (a counter). It is excluded when we ask "did the objects return to their start position?".
COUNTER_COL = 63


def play_area(frame):
    return frame[:, :COUNTER_COL]


def counter_cells(frame, initial):
    return int((frame[:, COUNTER_COL] != initial[:, COUNTER_COL]).sum())

def change_summary(before, after):
    # Factual description of what changed between two frames.
    if before.shape != after.shape:
        return {"shape_changed": True, "before_shape": list(before.shape), "after_shape": list(after.shape),
                "changed_cells": None, "bbox_xyxy": None, "color_transitions": {}}
    diff = before != after
    n = int(diff.sum())
    bbox = None
    transitions = {}
    if n:
        ys, xs = np.where(diff)
        bbox = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]
        pairs = Counter(zip(before[diff].tolist(), after[diff].tolist()))
        transitions = {f"{a}->{b}": c for (a, b), c in pairs.most_common()}
    return {"shape_changed": False, "changed_cells": n, "bbox_xyxy": bbox, "color_transitions": transitions}


def find_objects(frame, background=None, min_size=1):
    # 4-connected components of identical colour, excluding the background (most frequent colour).
    if background is None:
        background = Counter(frame.ravel().tolist()).most_common(1)[0][0]
    h, w = frame.shape
    seen = np.zeros_like(frame, dtype=bool)
    objects = []
    for y in range(h):
        for x in range(w):
            if seen[y, x] or frame[y, x] == background:
                continue
            color = int(frame[y, x])
            q, cells = deque([(y, x)]), []
            seen[y, x] = True
            while q:
                cy, cx = q.popleft()
                cells.append((cy, cx))
                for ny, nx in ((cy + 1, cx), (cy - 1, cx), (cy, cx + 1), (cy, cx - 1)):
                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and frame[ny, nx] == color:
                        seen[ny, nx] = True
                        q.append((ny, nx))
            if len(cells) < min_size:
                continue
            ys = [c[0] for c in cells]
            xs = [c[1] for c in cells]
            # click point = the object cell closest to its centroid (guaranteed to be on the object)
            cy, cx = np.mean(ys), np.mean(xs)
            py, px = min(cells, key=lambda c: (c[0] - cy) ** 2 + (c[1] - cx) ** 2)
            objects.append({"color": color, "size": len(cells),
                            "bbox_xyxy": [min(xs), min(ys), max(xs), max(ys)],
                            "click_xy": {"x": int(px), "y": int(py)}})
    return background, objects

## 1. Load AR25 and display the initial screen

In [ ]:
arc = Arcade()
env = arc.make(game_id=GAME_ID, seed=SEED, save_recording=False, include_frame_data=True)
if env is None:
    raise RuntimeError("Could not create the AR25 environment")

obs0 = env.reset()
if obs0 is None:
    raise RuntimeError("Reset failed")

frame0 = last_frame(obs0)
np.save(FRAMES_DIR / "initial.npy", frame0)

AVAILABLE = [f"ACTION{a}" if isinstance(a, int) else str(a) for a in obs0.available_actions]
ENV_INFO = {
    "game_id": GAME_ID, "seed": SEED,
    "frame_count_on_reset": len(obs0.frame), "frame_shape": list(frame0.shape),
    "state": state_name(obs0), "levels_completed": obs0.levels_completed,
    "win_levels": obs0.win_levels, "available_actions": AVAILABLE,
}
print(json.dumps(ENV_INFO, indent=2))

fig, ax = plt.subplots(figsize=(5, 5))
show_frame(frame0, ax, "AR25 — initial frame (seed 0)")
fig.savefig(IMAGES_DIR / "initial.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
BACKGROUND, OBJECTS = find_objects(frame0)
print(f"Background colour (most frequent): {BACKGROUND}")
print(f"Colour histogram: {dict(Counter(frame0.ravel().tolist()).most_common())}\n")
print(f"{len(OBJECTS)} connected object(s) found:")
for i, o in enumerate(OBJECTS):
    print(f"  obj{i:02d} colour={o['color']:>2} size={o['size']:>4} bbox(x0,y0,x1,y1)={o['bbox_xyxy']} click={o['click_xy']}")

In [ ]:
# Sanity check: is reset deterministic? (if not, every comparison below must be read with care)
check = last_frame(env.reset())
RESET_DETERMINISTIC = bool(np.array_equal(check, frame0))
print("Two resets give identical frames:", RESET_DETERMINISTIC)

## 2. Experiment runner

`run_experiment(exp_id, question, actions)`:
1. resets the game (every experiment starts from the same state → controlled),
2. applies each action, saving **all** returned frames as `.npy`,
3. measures the change of every step (cells changed, bounding box, colour transitions) and the net change vs. the start,
4. appends one JSON line to `experiments.jsonl`, and to `unsuccessful_experiments.jsonl` when nothing visible happened,
   an error occurred, or the game ended,
5. saves a figure (start → each step → net difference).

In [ ]:
RESULTS = {}


def _append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


def run_experiment(exp_id, question, actions, show=True, max_panels=6):
    obs = env.reset()
    start = last_frame(obs)
    np.save(FRAMES_DIR / f"{exp_id}_step00_start.npy", start)
    steps, error, prev = [], None, obs
    snapshots = [("start", start)]

    for i, (name, data) in enumerate(actions, start=1):
        if state_name(prev) in TERMINAL_STATES:
            error = f"stopped before step {i}: terminal state {state_name(prev)}"
            break
        before = last_frame(prev)
        try:
            res = env.step(GameAction[name], data=data or {})
        except Exception as exc:  # recorded, not hidden
            error = f"step {i} {name} {data}: {type(exc).__name__}: {exc}"
            break
        if res is None:
            error = f"step {i} {name} {data}: env.step returned None"
            break
        files = []
        for k, fr in enumerate(res.frame):
            fn = f"{exp_id}_step{i:02d}_after_{k:02d}.npy"
            np.save(FRAMES_DIR / fn, np.array(fr))
            files.append(fn)
        after = last_frame(res)
        steps.append({
            "step": i, "action": name, "action_data": data or {},
            "after_files": files, "after_frame_count": len(res.frame),
            "state_before": state_name(prev), "state_after": state_name(res),
            "levels_completed": res.levels_completed,
            "counter_cells_used": counter_cells(after, start),
            "play_area_changed_cells": change_summary(play_area(before), play_area(after))["changed_cells"],
            "available_actions": [f"ACTION{a}" if isinstance(a, int) else str(a) for a in res.available_actions],
            "full_reset": getattr(res, "full_reset", None),
            **change_summary(before, after),
        })
        snapshots.append((f"{i}: {name}" + (f" {data}" if data else ""), after))
        prev = res

    final = last_frame(prev)
    net = change_summary(start, final)
    net_play = change_summary(play_area(start), play_area(final))
    final_state = state_name(prev)
    total_changed = sum((s["changed_cells"] or 0) for s in steps)

    if error:
        outcome = "error"
    elif final_state == "GAME_OVER":
        outcome = "game_over"
    elif final_state == "WIN" or (steps and steps[-1]["levels_completed"] > obs.levels_completed):
        outcome = "level_progress"
    elif total_changed == 0:
        outcome = "no_visible_change"
    elif net_play["changed_cells"] == 0:
        outcome = "changed_then_returned_to_start"
    else:
        outcome = "changed"

    record = {
        "run_id": RUN_ID, "experiment_id": exp_id, "question": question,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "game_id": GAME_ID, "seed": SEED,
        "actions": [{"action": a, "action_data": d or {}} for a, d in actions],
        "steps_executed": len(steps), "steps": steps,
        "net_change": net, "net_change_play_area": net_play,
        "counter_cells_used": counter_cells(final, start), "final_state": final_state, "outcome": outcome, "error": error,
    }
    np.save(FRAMES_DIR / f"{exp_id}_final.npy", final)
    _append_jsonl(EXPERIMENTS_LOG, record)
    if outcome in {"error", "no_visible_change", "game_over"}:
        _append_jsonl(FAILED_LOG, {k: record[k] for k in
                                   ("run_id", "experiment_id", "question", "actions", "outcome", "error", "final_state")})
    RESULTS[exp_id] = record

    # figure
    shown = snapshots if len(snapshots) <= max_panels else snapshots[:max_panels - 1] + [snapshots[-1]]
    fig, axes = plt.subplots(1, len(shown) + 1, figsize=(2.6 * (len(shown) + 1), 2.9))
    for ax, (title, fr) in zip(axes, shown):
        show_frame(fr, ax, title)
    show_diff(start, final, axes[-1], f"net change: {net['changed_cells']} cells")
    fig.suptitle(f"{exp_id} — {outcome}", fontsize=10)
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / f"{exp_id}.png", dpi=110, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    per_step = [s["changed_cells"] for s in steps]
    print(f"{exp_id}: outcome={outcome} | changed cells per step={per_step} | net={net['changed_cells']} "
          f"| net bbox={net['bbox_xyxy']} | objects net={net_play['changed_cells']} "
          f"| counter cells={counter_cells(final, start)} | final state={final_state}" + (f" | ERROR: {error}" if error else ""))
    return record


SIMPLE_ACTIONS = [a for a in ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"] if a in AVAILABLE]
print("Simple actions available:", SIMPLE_ACTIONS, "| ACTION6 (click) available:", "ACTION6" in AVAILABLE)

## 3. Controlled experiments

Each experiment changes **one** thing compared with the reset state.

### E01 — Each simple action once
*Question:* what does each non-click action do from the initial state?

In [ ]:
for a in SIMPLE_ACTIONS:
    run_experiment(f"E01_{a}", f"What does {a} do once from the initial state?", [(a, {})])

### E02 — Repetition (×3)
*Question:* is the effect of an action cumulative (same change repeated), saturating, or a toggle?

In [ ]:
for a in SIMPLE_ACTIONS:
    run_experiment(f"E02_{a}_x3", f"Is {a} cumulative when repeated 3 times?", [(a, {})] * 3)

### E03 — Action followed by its candidate opposite
*Question:* does the second action undo the first (net change = 0 → `changed_then_returned_to_start`)?

In [ ]:
PAIRS = [("ACTION1", "ACTION2"), ("ACTION2", "ACTION1"), ("ACTION3", "ACTION4"), ("ACTION4", "ACTION3")]
for a, b in PAIRS:
    if a in AVAILABLE and b in AVAILABLE:
        run_experiment(f"E03_{a}_then_{b}", f"Does {b} undo {a}?", [(a, {}), (b, {})])

### E04 — `ACTION6` click on each object and on empty background
*Question:* which screen elements react to a click? Does clicking empty background do anything?

In [ ]:
if "ACTION6" in AVAILABLE:
    for i, o in enumerate(OBJECTS):
        run_experiment(f"E04_click_obj{i:02d}_c{o['color']}",
                       f"What happens when clicking object obj{i:02d} (colour {o['color']}) at {o['click_xy']}?",
                       [("ACTION6", o["click_xy"])], show=len(OBJECTS) <= 12)
    # background points: pick cells of the background colour, far from any object
    h, w = frame0.shape
    candidates = [(5, 5), (w // 2 - 6, h - 8), (w - 8, h // 2)]
    for j, (x, y) in enumerate(candidates):
        if frame0[y, x] == BACKGROUND:
            run_experiment(f"E04_click_background_{j}", f"Does clicking empty background at x={x},y={y} change anything?",
                           [("ACTION6", {"x": x, "y": y})])
else:
    print("ACTION6 not available — E04 skipped")

### E05 — Determinism
*Question:* does the same action sequence from reset always give the same final frame?

In [ ]:
DETERMINISM = {}
for a in SIMPLE_ACTIONS[:4]:
    seq = [(a, {}), (a, {})]
    r1 = run_experiment(f"E05_{a}_run1", f"Determinism check for {a}x2 (run 1)", seq, show=False)
    r2 = run_experiment(f"E05_{a}_run2", f"Determinism check for {a}x2 (run 2)", seq, show=False)
    f1 = np.load(FRAMES_DIR / f"E05_{a}_run1_final.npy")
    f2 = np.load(FRAMES_DIR / f"E05_{a}_run2_final.npy")
    DETERMINISM[a] = bool(np.array_equal(f1, f2))
print("Identical final frames per action:", DETERMINISM)

### E06 — Click an object, then move
*Question:* does clicking an object change which object the movement actions affect?
Compare with E01 (same movement without a prior click).

In [ ]:
MOVE = "ACTION1" if "ACTION1" in AVAILABLE else (SIMPLE_ACTIONS[0] if SIMPLE_ACTIONS else None)
if "ACTION6" in AVAILABLE and MOVE:
    # the largest objects are likely the interesting ones; limit to 4 to keep the run short
    for i, o in sorted(enumerate(OBJECTS), key=lambda t: -t[1]["size"])[:4]:
        run_experiment(f"E06_click_obj{i:02d}_then_{MOVE}",
                       f"After clicking obj{i:02d} (colour {o['color']}), what does {MOVE} move?",
                       [("ACTION6", o["click_xy"]), (MOVE, {})])
else:
    print("E06 skipped (needs ACTION6 and a movement action)")

### E07 — Seeded random walk (30 steps)
*Question:* over a longer sequence, do we ever see a level change, a game over, or a full reset?
The random generator is seeded (`RANDOM_SEED`), so the sequence is reproducible.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
walk = []
for _ in range(30):
    a = str(rng.choice(AVAILABLE))
    if a == "ACTION6":
        walk.append((a, {"x": int(rng.integers(0, frame0.shape[1])), "y": int(rng.integers(0, frame0.shape[0]))}))
    else:
        walk.append((a, {}))
r = run_experiment("E07_random_walk_30", "Does a 30-step seeded random walk reach a level change or game over?", walk)
for s in r["steps"]:
    flags = []
    if s["state_after"] != s["state_before"]:
        flags.append(f"STATE {s['state_before']}->{s['state_after']}")
    if s["full_reset"]:
        flags.append("FULL_RESET")
    if s["after_frame_count"] > 1:
        flags.append(f"{s['after_frame_count']} frames")
    print(f"  {s['step']:>2} {s['action']:<8} {str(s['action_data']):<20} changed={s['changed_cells']} levels={s['levels_completed']} {' '.join(flags)}")

### E08 — Is `ACTION7` an undo?
*Observed in run 20261120T012831Z:* `ACTION7` changes nothing from the start (E01), but changed 108 cells in the middle
of the random walk (E07). *Question:* does `ACTION7` revert the previous move?

In [ ]:
E08 = [
    ("E08_ACTION1_then_ACTION7", [("ACTION1", {}), ("ACTION7", {})]),
    ("E08_ACTION3_then_ACTION7", [("ACTION3", {}), ("ACTION7", {})]),
    ("E08_ACTION1_ACTION1_then_ACTION7_x2", [("ACTION1", {}), ("ACTION1", {}), ("ACTION7", {}), ("ACTION7", {})]),
    ("E08_ACTION5_then_ACTION7", [("ACTION5", {}), ("ACTION7", {})]),
]
for eid, seq in E08:
    if all(a in AVAILABLE for a, _ in seq):
        r = run_experiment(eid, "Does ACTION7 undo the previous action(s)?", seq)
        print("   play-area cells changed per step:", [s["play_area_changed_cells"] for s in r["steps"]],
              "| counter after each step:", [s["counter_cells_used"] for s in r["steps"]])

### E09 — Move the yellow L onto the static L
*Observed in run 20261120T012831Z:* `ACTION1`/`ACTION2` move the dotted L and the yellow L up/down by 3 cells;
`ACTION3`/`ACTION4` move them horizontally in **opposite** directions (mirror). Another L of the same shape
(colour 11, bbox x51–59, y45–53) never moved.
*Question (tests H7):* if the yellow L is placed exactly on that static L, does the level complete?
Offsets are computed from the current frame, not hard-coded.

In [ ]:
def bbox_of(frame, color):
    ys, xs = np.where(play_area(frame) == color)
    return (int(xs.min()), int(ys.min())) if len(xs) else None

f = last_frame(env.reset())
yellow = bbox_of(f, 4)      # colour 4 = yellow L (moves)
target = next(o for o in OBJECTS if o["color"] == 11 and o["size"] == 45)  # static L
tx, ty = target["bbox_xyxy"][0], target["bbox_xyxy"][1]
dx, dy = tx - yellow[0], ty - yellow[1]
print(f"yellow L at {yellow}, static L at {(tx, ty)}, offset dx={dx}, dy={dy}")

if dx % 3 or dy % 3:
    print("Offset is not a multiple of 3 cells — cannot be aligned with 3-cell moves; E09 skipped.")
else:
    # ACTION3 moves the yellow L right, ACTION4 left; ACTION2 down, ACTION1 up (E01 run 20261120T012831Z)
    seq = ([("ACTION2", {})] * (dy // 3) if dy > 0 else [("ACTION1", {})] * (-dy // 3)) + \
          ([("ACTION3", {})] * (dx // 3) if dx > 0 else [("ACTION4", {})] * (-dx // 3))
    r = run_experiment("E09_yellow_onto_static_L", "Does placing the yellow L on the static L complete the level?", seq)
    last = r["steps"][-1] if r["steps"] else {}
    print("levels_completed after last step:", last.get("levels_completed"), "| state:", last.get("state_after"),
          "| full_reset:", last.get("full_reset"), "| frames returned:", last.get("after_frame_count"))
    # variant: vertical moves last, in case the order matters
    seq2 = seq[len(seq) - abs(dx) // 3:] + seq[:len(seq) - abs(dx) // 3]
    run_experiment("E09b_yellow_onto_static_L_horizontal_first", "Same target, horizontal moves first", seq2)

## 4. Observations vs. hypotheses

**Observations** below are computed directly from the logged data — they are facts about this run.

**Hypotheses** are interpretations. Each one has a status:
`untested` · `supported` · `refuted` · `inconclusive`. Some are evaluated automatically from the experiments above;
the others must be updated **by hand after looking at the figures** — never mark one `supported` without citing evidence.

In [ ]:
# ---------- automatic observations (facts) ----------
OBSERVATIONS = []
for exp_id, r in RESULTS.items():
    if exp_id.startswith(("E01_", "E04_")):
        s = r["steps"][0] if r["steps"] else None
        if s:
            OBSERVATIONS.append(
                f"{exp_id}: {s['action']} {s['action_data'] or ''} changed {s['changed_cells']} cells "
                f"in bbox {s['bbox_xyxy']}; colour transitions {dict(list(s['color_transitions'].items())[:4])}")
OBSERVATIONS.append(f"Reset is deterministic: {RESET_DETERMINISTIC}")
OBSERVATIONS.append(f"Same sequence → same final frame: {DETERMINISM}")
for o in OBSERVATIONS:
    print("•", o)

In [ ]:
# ---------- hypotheses ----------
def evidence_of(exp_id):
    r = RESULTS.get(exp_id)
    return f"{exp_id}: outcome={r['outcome']}, net change={r['net_change']['changed_cells']}" if r else f"{exp_id}: not run"


HYPOTHESES = []

# H1 — automatic: opposite action pairs
for a, b in PAIRS:
    eid = f"E03_{a}_then_{b}"
    if eid in RESULTS:
        r = RESULTS[eid]
        status = ("supported" if r["outcome"] == "changed_then_returned_to_start"
                  else "inconclusive" if r["outcome"] in {"no_visible_change", "error"} else "refuted")
        HYPOTHESES.append({"id": f"H1_{a}_{b}", "statement": f"{b} brings the objects back after {a} (counter column ignored)",
                           "status": status,
                           "evidence": f"{evidence_of(eid)}; objects net change={r['net_change_play_area']['changed_cells']}"})

# H2 — automatic: clicking empty background has no effect
bg = [k for k in RESULTS if k.startswith("E04_click_background")]
if bg:
    no_effect = all(RESULTS[k]["outcome"] == "no_visible_change" for k in bg)
    HYPOTHESES.append({"id": "H2", "statement": "Clicking empty background has no visible effect",
                       "status": "supported" if no_effect else "refuted",
                       "evidence": "; ".join(evidence_of(k) for k in bg)})

# H3 — automatic: determinism
if DETERMINISM:
    HYPOTHESES.append({"id": "H3", "statement": "The game is deterministic for a fixed seed and action sequence",
                       "status": "supported" if all(DETERMINISM.values()) else "refuted",
                       "evidence": f"E05 identical finals: {DETERMINISM}; reset identical: {RESET_DETERMINISTIC}"})

# H4 — automatic: repetition is cumulative (each of the 3 repeats changes the frame)
for a in SIMPLE_ACTIONS:
    eid = f"E02_{a}_x3"
    if eid in RESULTS:
        per = [s["changed_cells"] for s in RESULTS[eid]["steps"]]
        status = ("supported" if len(per) == 3 and all(per) and RESULTS[eid]["net_change"]["changed_cells"]
                  else "refuted" if per and per[0] and not all(per) else "inconclusive")
        HYPOTHESES.append({"id": f"H4_{a}", "statement": f"Repeating {a} keeps changing the frame (cumulative effect)",
                           "status": status, "evidence": f"{eid}: changed cells per step {per}"})

# H5+ — set from the figures/frames of run 20261120T012831Z. Re-check them when you re-run.
RUN1 = "run 20261120T012831Z"
HYPOTHESES += [
    {"id": "H5", "statement": "ACTION1/ACTION2 move both the dotted L and the yellow L up/down by 3 cells",
     "status": "supported", "evidence": f"{RUN1}, E01: both bboxes y 16→13 (ACTION1) and 16→19 (ACTION2); x unchanged"},
    {"id": "H5b", "statement": "ACTION3/ACTION4 move the two Ls horizontally by 3 cells in opposite directions "
     "(mirror around the white bar); ACTION3 = dotted L left / yellow L right",
     "status": "supported", "evidence": f"{RUN1}, E01: ACTION3 dotted x 19→16, yellow x 36→39; ACTION4 the reverse"},
    {"id": "H5c", "statement": "The yellow L is the mirror image of the dotted L across the white bar (x=30–32)",
     "status": "supported", "evidence": f"{RUN1}, initial frame: dotted x18–26, yellow x36–44 (symmetric around x=31), shapes mirrored"},
    {"id": "H5d", "statement": "The Ls can cross the white bar (they are not blocked by it)",
     "status": "supported", "evidence": f"{RUN1}, E02_ACTION4_x3 step 3: dotted L at x26–34, yellow L at x27–29, overlapping the bar"},
    {"id": "H6", "statement": "Clicking an object selects it, and movement actions then apply to the selected object",
     "status": "refuted", "evidence": f"{RUN1}, E04: no click changed anything; E06: ACTION1 after any click = same 109-cell change as E01"},
]

# H7 — automatic from E09 / E09b
e9 = [RESULTS[k] for k in ("E09_yellow_onto_static_L", "E09b_yellow_onto_static_L_horizontal_first") if k in RESULTS]
if e9:
    ok = all(r["outcome"] == "level_progress" for r in e9)
    ev = "; ".join(f"{r['experiment_id']}: outcome={r['outcome']}, level reached at step {next((s['step'] for s in r['steps'] if s['levels_completed'] > 0), None)}/{len(r['actions'])}" for r in e9)
    HYPOTHESES.append({"id": "H7", "statement": "Placing the yellow L exactly on the static L (colour 11) completes level 1, whatever the move order",
                       "status": "supported" if ok else "refuted", "evidence": ev})

# H8 — automatic from E08 (objects back to start after undo; counter column ignored)
e8 = [RESULTS[k] for k in RESULTS if k.startswith("E08_") and "ACTION5" not in k]
if e8:
    ok = all(r["net_change_play_area"]["changed_cells"] == 0 and r["outcome"] == "changed_then_returned_to_start" for r in e8)
    HYPOTHESES.append({"id": "H8", "statement": "ACTION7 undoes the previous move (one move per ACTION7)",
                       "status": "supported" if ok else "refuted",
                       "evidence": "; ".join(f"{r['experiment_id']}: objects net change={r['net_change_play_area']['changed_cells']}" for r in e8)})
    counters = {r["experiment_id"]: [s["counter_cells_used"] for s in r["steps"]] for r in e8}
    refunded = any(c[-1] < max(c) for c in counters.values())
    HYPOTHESES.append({"id": "H8b", "statement": "ACTION7 gives back the counter cell of the undone move",
                       "status": "supported" if refunded else "refuted", "evidence": f"counter after each step: {counters}"})

HYPOTHESES += [
    {"id": "H9", "statement": "Column x=63 is a move counter / budget: one cell turns 11→5 per action (ACTION1–5); clicks do not count",
     "status": "supported", "evidence": f"{RUN1}: E01 ACTION1–5 each change exactly 1 cell in column 63; E04 clicks change 0"},
]

for h in HYPOTHESES:
    print(f"[{h['status']:>12}] {h['id']}: {h['statement']}\n               evidence: {h['evidence']}")

## 5. Unsuccessful experiments (no visible effect, error, or game over)

In [ ]:
failed = [r for r in RESULTS.values() if r["outcome"] in {"error", "no_visible_change", "game_over"}]
print(f"{len(failed)} unsuccessful experiment(s) — also saved in {FAILED_LOG.relative_to(ROOT)}\n")
for r in failed:
    acts = ", ".join(f"{a['action']}{a['action_data'] or ''}" for a in r["actions"][:5])
    print(f"• {r['experiment_id']}: {r['outcome']} — {acts}" + (f" — {r['error']}" if r["error"] else ""))

## 6. Summary and report

Fill in `DISCOVERIES` and `OPEN_QUESTIONS` **after** looking at the figures, then run the last cell.
Write discoveries as facts that cite an experiment ID; anything you are not sure of belongs in a hypothesis or a question.

In [ ]:
DISCOVERIES = [
    "Reset and action sequences are deterministic (E05).",
    "Two objects are controlled together: the dotted L (left half) and the yellow L (right half), which mirror each other across the white bar.",
    "ACTION1 = up 3 cells, ACTION2 = down 3 cells for both Ls; ACTION3 = dotted L left / yellow L right, ACTION4 = the opposite (E01, E02).",
    "The Ls pass through the white bar and can swap sides (E02_ACTION4_x3).",
    "Column x=63 fills one cell per move: it behaves like a move counter (E01, E03). It resets to empty when a level is completed (E09).",
    "ACTION5 only advances the counter: no other visible effect (E01_ACTION5, E02_ACTION5_x3).",
    "ACTION6 (click) never changed anything on level 1, on objects or background, and does not advance the counter (E04, E06).",
    "ACTION7 undoes the last move (E08). It does not advance the counter, and does not give the counter cell back.",
    "LEVEL 1 SOLVED: moving the yellow L exactly onto the static L of the same shape completes level 1 in 15 moves "
    "(10 x ACTION2 + 5 x ACTION3, in either order) — levels_completed 0 → 1, 3-frame transition, new level shown (E09, E09b).",
    "30 seeded random actions did not complete a level nor end the game (E07).",
]

OPEN_QUESTIONS = [
    "What does level 2 look like, and does the same rule (move the mirrored shape onto the static target) apply?",
    "What happens when the counter column is full (64 moves)? Game over or level reset?",
    "Does the dotted L have any role besides mirroring the yellow L (e.g. obstacles in later levels)?",
    "Is ACTION5 useful later (e.g. a 'wait' action), and does ACTION6 do anything in later levels?",
    "Level-1 human baseline is 32 actions (env.info.baseline_actions); our 15-move solution is shorter — confirm what the scoring counts.",
]

In [ ]:
def md_table(rows, headers):
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    out += ["| " + " | ".join(str(c).replace("|", "/") for c in row) + " |" for row in rows]
    return "\n".join(out)


exp_rows = []
for r in RESULTS.values():
    acts = ", ".join(f"{a['action']}{a['action_data'] or ''}" for a in r["actions"][:4])
    if len(r["actions"]) > 4:
        acts += f", … ({len(r['actions'])} total)"
    exp_rows.append([r["experiment_id"], acts, [s["changed_cells"] for s in r["steps"]][:6],
                     r["net_change"]["changed_cells"], r["net_change"]["bbox_xyxy"], r["outcome"]])

report = f'''# AR25 exploration report — Teammate 2

- **Run ID:** `{RUN_ID}`
- **Game:** `{GAME_ID}`, seed {SEED}
- **Raw data:** `{OUT.relative_to(ROOT).as_posix()}/` (frames `.npy`, figures `.png`, `experiments.jsonl`, `unsuccessful_experiments.jsonl`)
- **Reproduce:** run `notebooks/teammate_2_exploration.ipynb` top to bottom (random walk seed = {RANDOM_SEED}).

## Environment
```json
{json.dumps(ENV_INFO, indent=2)}
```

Background colour: {BACKGROUND}. Objects detected on the initial frame: {len(OBJECTS)}.

{md_table([[f"obj{i:02d}", o["color"], o["size"], o["bbox_xyxy"], o["click_xy"]] for i, o in enumerate(OBJECTS)],
          ["object", "colour", "cells", "bbox (x0,y0,x1,y1)", "click point"])}

## Experiments ({len(RESULTS)})
{md_table(exp_rows, ["id", "actions", "changed cells per step", "net change", "net bbox", "outcome"])}

## Observations (measured)
{chr(10).join("- " + o for o in OBSERVATIONS)}

## Hypotheses
{md_table([[h["id"], h["statement"], h["status"], h["evidence"]] for h in HYPOTHESES], ["id", "hypothesis", "status", "evidence"])}

## Unsuccessful experiments ({len(failed)})
{chr(10).join(f"- `{r['experiment_id']}` — {r['outcome']}" + (f" — {r['error']}" if r["error"] else "") for r in failed) or "- none"}

## Discoveries
{chr(10).join("- " + d for d in DISCOVERIES) or "- _(not filled in yet)_"}

## Open questions
{chr(10).join("- " + q for q in OPEN_QUESTIONS)}
'''

report_path = REPORTS_DIR / "teammate_2_exploration_report.md"
report_path.write_text(report, encoding="utf-8")
print("Report written to", report_path.relative_to(ROOT))
print("Experiments:", len(RESULTS), "| unsuccessful:", len(failed), "| hypotheses:", len(HYPOTHESES))